- #### Libs & Functions 

In [1]:
# import packages
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, RocCurveDisplay
import numpy as np
from sklearn.metrics import precision_recall_curve, f1_score, precision_score, recall_score, confusion_matrix, classification_report

import json, re
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline


- ####  Global Parameters

- ####  Read Data

In [ ]:
path = "C:/Users/clamo/Documents/Doutorado/HC/hc_models/features/"

In [ ]:
# pip install -q transformers accelerate torch

# 1) Escolha um modelo open-source
model_id = "Qwen/Qwen2.5-1.5B-Instruct"  # ou "mistralai/Mistral-7B-Instruct-v0.3"

tok = AutoTokenizer.from_pretrained(model_id)
mdl = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto", torch_dtype="auto")
gen = pipeline("text-generation", model=mdl, tokenizer=tok)

# 2) Texto de exemplo (seu trecho)
texto = (
    "Paciente no 1 DIH por Miomatose uterina em pré operatório imediato, com estado geral regular, em glasgow 15, respirando em ar ambiente, "
    "tórax simétrico e expansível bilateralmente , dieta zero até segunda ordem, abdomen plano e flácido indolor à palpação , "
    "diurese e evacuações presentes sem alterações, ausência de sangramento vaginal, MMSSII perfundidos sem edemas. Paciente sem AVP.\r\n "
)

# 3) Prompt ultra simples pedindo apenas JSON
prompt = f"""
Você é um codificador clínico brasileiro. Atribua **códigos CID-10** ao texto abaixo.

Regras:
- Retorne SOMENTE um JSON válido, no formato:
  {{
    "cid": [ {{ "code": "CID", "confidence": 0.0 }} ]
  }}
- Use apenas CID-10 (não inclua procedimentos, cirurgias ou medicamentos).
- Se não houver diagnóstico, utilize códigos de sinais/sintomas (capítulo R).
- Não escreva nada fora do JSON.

TEXTO:
\"\"\"{texto}\"\"\"
JSON:
"""

out = gen(prompt, max_new_tokens=300, temperature=0.2, do_sample=True)[0]["generated_text"]

out

In [ ]:
out

In [ ]:
import re, json

# 1) tente capturar o bloco cercado por ```json ... ```
m = re.search(r"```json\s*(\{.*?\})\s*```", out, flags=re.S|re.I)
if m:
    payload = m.group(1)
else:
    # fallback: captura o primeiro objeto JSON não-guloso
    m = re.search(r"\{.*?\}", out, flags=re.S)
    payload = m.group(0) if m else None

resp = json.loads(payload) if payload else {"cids": [], "notes": "sem_json"}
print(json.dumps(resp, ensure_ascii=False, indent=2))


In [ ]:

# 4) Extrair o JSON (caso o modelo escreva algo a mais)
match = re.search(r"\{.*\}", out, flags=re.S)
resp = json.loads(match.group(0)) if match else {"cids": [], "notes": "sem_json"}

print(json.dumps(resp, ensure_ascii=False, indent=2))

In [ ]:
internacao = pd.read_parquet(path + "features_internacao.parquet").reset_index(drop=True)
targets = pd.read_parquet(path + "target_internacao.parquet").reset_index(drop=True)
evolucoes = pd.read_parquet(path + "embbeding_evolucao.parquet").reset_index(drop=True)
pacientes = pd.read_parquet(path + "pacientes.parquet").reset_index(drop=True)
exames = pd.read_parquet(path + "features_exames.parquet").reset_index(drop=True)
consultas = pd.read_parquet(path + "features_consultas.parquet").reset_index(drop=True)

In [ ]:

internacao["prontuario"] = internacao["prontuario"].astype("Int64").astype(str)
#targets["prontuario"] = targets["prontuario"].astype("Int64").astype(str)
#evolucoes["prontuario"] = evolucoes["prontuario"].astype("Int64").astype(str)
pacientes["prontuario"] = pacientes["prontuario"].astype("Int64").astype(str)
exames["prontuario"] = exames["prontuario"].astype("Int64").astype(str)
consultas["prontuario"] = consultas["prontuario"].astype("Int64").astype(str)

In [ ]:
# Converte targets para datetime
targets["date_ref"] = pd.to_datetime(targets["date_ref"], errors="coerce").dt.normalize()

# Garante que os outros também estejam normalizados (sem horas)
internacao["date_ref"] = pd.to_datetime(internacao["date_ref"], errors="coerce").dt.normalize()
#evolucoes["date_ref"] = pd.to_datetime(evolucoes["date_ref"], errors="coerce").dt.normalize()

In [ ]:
#evolucoes.sort_values(["prontuario", "date_ref"])

In [ ]:
data = targets.merge(pacientes, on=["prontuario"])
data.shape

In [ ]:
data= data.merge(internacao, on=["prontuario", "date_ref"],  how='left')
data.shape

In [ ]:
data= data.merge(exames, on=["prontuario", "date_ref"],  how='left')
data.shape

In [ ]:
data= data.merge(consultas, on=["prontuario", "date_ref"],  how='left')
data.shape

In [ ]:
# Converte targets para datetime
#targets["date_ref"] = pd.to_datetime(targets["date_ref"], errors="coerce").dt.normalize()

# Garante que os outros também estejam normalizados (sem horas)
#internacao["date_ref"] = pd.to_datetime(internacao["date_ref"], errors="coerce").dt.normalize()
#evolucoes["date_ref"] = pd.to_datetime(evolucoes["date_ref"], errors="coerce").dt.normalize()

In [ ]:
targets[["mais_de_7_dias"]].groupby("mais_de_7_dias").size()

In [ ]:
targets[["mais_de_15_dias"]].groupby("mais_de_15_dias").size()

In [ ]:
# pd show all rows
pd.set_option('display.max_rows', None)

In [ ]:
# total rows
total_rows = len(data)

# count nulls and percentages
null_counts = data.isnull().sum()
null_percent = (null_counts / total_rows) * 100

# combine into one DataFrame
null_stats = pd.DataFrame({
    "null_count": null_counts,
    "null_percent": null_percent
}).reset_index()

print(null_stats)

<h3> Split data <h3>

In [ ]:
# remove agosto de 2021 pra tras por conta da build dos dados
#data = data[data["date_ref"] >= "2023-07-01"]

In [ ]:
# Calculando a porcentagem de nulos para cada coluna de `data`
null_percent = data.isnull().mean() * 100

# Identificando as colunas com mais de 90% de nulos
cols_to_drop = null_percent[null_percent > 95].index

# Removendo essas colunas do DataFrame `data`
data = data.drop(columns=cols_to_drop)

# Exibindo as colunas removidas e o novo DataFrame
print("Colunas removidas:", cols_to_drop)

In [ ]:
#list(data.columns)

In [ ]:
target_column = 'mais_de_15_dias'

In [ ]:
nowanted_cols = ['prontuario', 'date_ref', 'dias_internado', 'mais_de_4_dias', 'mais_de_7_dias', 'mais_de_15_dias']

feature_columns = [item for item in list(data.columns) if item not in nowanted_cols]


In [ ]:
# split for time evaluation
target_time = data[data["date_ref"] >= "2025-06-01"]['mais_de_7_dias']
target = data[data["date_ref"] < "2025-06-01"]['mais_de_7_dias']

#
features_time = data[data["date_ref"] >= "2025-06-01"][feature_columns]
features = data[data["date_ref"] < "2025-06-01"][feature_columns]

In [ ]:

# 1. Remove colunas que não são numéricas ou booleanas
#train = train.select_dtypes(include=["number", "bool"]).copy()

# 2. Substitui infinitos por NaN
#train = train.replace([np.inf, -np.inf], np.nan)

# --- data ---
# df = pd.read_csv("data.csv")
#target = "target_180_empretec"
#X = train.drop(columns=[target] + ([ 'date_ref'] if 'date_ref' in train.columns else [])).copy()
#y = train[target].astype(int)

#X = X.select_dtypes(include=["number", "bool"]).copy()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(features, target, test_size=0.15, random_state=42)

In [ ]:
# --- model ---
clf = LGBMClassifier(random_state=42)
clf.fit(X_train, y_train)

#### eval o time

In [ ]:

# --- AUROC ---
proba = clf.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, proba)
print(f"AUROC: {auc:.4f}")

# --- ROC curve (optional) ---
RocCurveDisplay.from_predictions(y_test, proba)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# --- Gráfico de densidade dos scores preditos ---
plt.figure(figsize=(8, 5))
sns.kdeplot(proba[y_test == 0], label="Classe 0", fill=True)
sns.kdeplot(proba[y_test == 1], label="Classe 1", fill=True)
plt.title("Distribuição das probabilidades preditas (densidade)")
plt.xlabel("Probabilidade prevista da classe positiva")
plt.ylabel("Densidade")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.3)
plt.show()


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    PrecisionRecallDisplay
)

# --- Predictions ---
preds = clf.predict(X_test)
proba = clf.predict_proba(X_test)[:, 1]

# --- Metrics ---
auc = roc_auc_score(y_test, proba)
acc = accuracy_score(y_test, preds)
prec = precision_score(y_test, preds, zero_division=0)
rec = recall_score(y_test, preds)
f1 = f1_score(y_test, preds)

print(f"AUROC:   {auc:.4f}")
print(f"Accuracy:{acc:.4f}")
print(f"Precision:{prec:.4f}")
print(f"Recall:  {rec:.4f}")
print(f"F1-score:{f1:.4f}")

# --- Confusion Matrix ---
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, preds))

# --- Full Classification Report ---
print("\nClassification Report:")
print(classification_report(y_test, preds, digits=4))

# --- Precision-Recall Curve (useful for imbalance) ---
PrecisionRecallDisplay.from_predictions(y_test, proba)


In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Compute confusion matrix
cm = confusion_matrix(y_test, preds)

# Create a plot
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap="Blues", values_format="d")

plt.title("Confusion Matrix")
plt.show()

#### eval out of time

In [ ]:

# --- AUROC ---
proba = clf.predict_proba(features_time)[:, 1]
auc = roc_auc_score(target_time, proba)
print(f"AUROC: {auc:.4f}")

# --- ROC curve (optional) ---
RocCurveDisplay.from_predictions(target_time, proba)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# --- Gráfico de densidade dos scores preditos ---
plt.figure(figsize=(8, 5))
sns.kdeplot(proba[target_time == 0], label="Classe 0", fill=True)
sns.kdeplot(proba[target_time == 1], label="Classe 1", fill=True)
plt.title("Distribuição das probabilidades preditas (densidade)")
plt.xlabel("Probabilidade prevista da classe positiva")
plt.ylabel("Densidade")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.3)
plt.show()

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    PrecisionRecallDisplay
)

# --- Predictions ---
preds = clf.predict(features_time)
proba = clf.predict_proba(features_time)[:, 1]

# --- Metrics ---
auc = roc_auc_score(target_time, proba)
acc = accuracy_score(target_time, preds)
prec = precision_score(target_time, preds, zero_division=0)
rec = recall_score(target_time, preds)
f1 = f1_score(target_time, preds)

print(f"AUROC:   {auc:.4f}")
print(f"Accuracy:{acc:.4f}")
print(f"Precision:{prec:.4f}")
print(f"Recall:  {rec:.4f}")
print(f"F1-score:{f1:.4f}")

# --- Confusion Matrix ---
print("\nConfusion Matrix:")
print(confusion_matrix(target_time, preds))

# --- Full Classification Report ---
print("\nClassification Report:")
print(classification_report(target_time, preds, digits=4))

# --- Precision-Recall Curve (useful for imbalance) ---
PrecisionRecallDisplay.from_predictions(target_time, proba)

In [ ]:
#false_negatives = X_test[(preds == 0) & (y_test == 1)]
#false_negatives

In [ ]:

# proba = clf.predict_proba(X_test)[:, 1]
# y_test = target_time

# --- 1) Threshold que maximiza F1 ---
prec, rec, thr = precision_recall_curve(target_time, proba)
# obs: len(thr) = len(prec) - 1 = len(rec) - 1
f1_vals = (2 * prec[1:] * rec[1:]) / (prec[1:] + rec[1:] + 1e-12)
i_best = np.argmax(f1_vals)
best_thr = thr[i_best]

preds_best = (proba >= best_thr).astype(int)
print(f"[F1 Máximo] threshold = {best_thr:.4f}")
print(f"Precision={precision_score(target_time, preds_best, zero_division=0):.4f} | "
      f"Recall={recall_score(target_time, preds_best):.4f} | "
      f"F1={f1_score(target_time, preds_best):.4f}")
print("Confusion Matrix:\n", confusion_matrix(target_time, preds_best))
print("\nClassification Report:\n", classification_report(target_time, preds_best, digits=4))

# --- 2) Threshold com recall mínimo (ex.: >= 0.70) e melhor F1 entre eles ---
recall_min = 0.70
candidates = np.where(rec[1:] >= recall_min)[0]
if len(candidates) > 0:
    j = candidates[np.argmax(f1_vals[candidates])]
    thr_rec = thr[j]
    preds_rec = (proba >= thr_rec).astype(int)
    print(f"\n[Recall ≥ {recall_min:.2f}] threshold = {thr_rec:.4f}")
    print(f"Precision={precision_score(target_time, preds_rec, zero_division=0):.4f} | "
          f"Recall={recall_score(target_time, preds_rec):.4f} | "
          f"F1={f1_score(target_time, preds_rec):.4f}")
    print("Confusion Matrix:\n", confusion_matrix(target_time, preds_rec))
else:
    print(f"\nNenhum threshold alcança recall ≥ {recall_min:.2f}. "
          "Tente reduzir o limite ou ajustar class_weight/params do modelo.")
